## Set Up

In [ ]:
!pip install -q transformers==4.48.0 datasets wandb sentencepiece accelerate lm-eval -U

In [ ]:
import os
import random
import numpy as np
import torch

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from transformers import logging
logging.set_verbosity_error()

## Constants

In [ ]:
SEED: int = 1

PRUNING_RATIO: float = 0.5
CALIBRATION_SAMPLES: int = 2048
ROUTER_TRAIN_SAMPLES: int = 1024
SPLIT_LAYER: int = 15  # Layer index where wiki expert ends and dataset expert starts
BATCH_SIZE: int = 64

EVAL_LOG_FILE: str = "logs/eval_runs.jsonl"
LLM_MODEL: str = "huggyllama/llama-7b"
DEVICE: str = "cuda:0"

EXPERT_DIR: str = "experts"
DATA_DIR: str = "adaptive_pruning/data"
ROUTER_HIDDEN_DIR: str = "adaptive_pruning/data/router_training"
ROUTER_MODEL_PATH: str = "adaptive_pruning/router_model.pt"
os.makedirs(EXPERT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ROUTER_HIDDEN_DIR, exist_ok=True)
os.makedirs(os.path.dirname(EVAL_LOG_FILE) or ".", exist_ok=True)

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 1) Extract FLAP masks

In [ ]:
import subprocess

extract_specs = [
    ("wikitext2", os.path.join(EXPERT_DIR, "wiki.pt")),
    ("boolq", os.path.join(EXPERT_DIR, "boolq.pt")),
    ("arc_easy", os.path.join(EXPERT_DIR, "arc_easy.pt")),
    ("winogrande", os.path.join(EXPERT_DIR, "winogrande.pt")),
    ("hellaswag", os.path.join(EXPERT_DIR, "hellaswag.pt")),
]

for dataset, out_path in extract_specs:
    subprocess.run(
        [
            "python",
            "-m",
            "adaptive_pruning.run_extract",
            "--model",
            LLM_MODEL,
            "--calibration_dataset",
            dataset,
            "--pruning_ratio",
            str(PRUNING_RATIO),
            "--remove_heads",
            "-1",
            "--metrics",
            "WIFV",
            "--structure",
            "AL-AM",
            "--nsamples",
            str(CALIBRATION_SAMPLES),
            "--seed",
            str(SEED),
            "--output",
            out_path,
        ],
        check=True,
    )

## 2) Router training

### 2.1 Collect router training data

In [ ]:
train_datasets = ["arc_easy", "boolq", "winogrande", "hellaswag", "arc_challenge"]

for dataset in train_datasets:
    subprocess.run(
        [
            "python",
            "-m",
            "adaptive_pruning.router.collect_training_data",
            "--dataset",
            dataset,
            "--output",
            os.path.join(DATA_DIR, dataset),
        ],
        check=True,
    )

### 2.2 Collect hidden states

In [ ]:
hidden_specs = [
    {
        "expert": os.path.join(EXPERT_DIR, "wiki_arc.pt"),
        "pairs": [
            ("mom/new/data/arc_easy_train.pt", "hidden_logprobs_wiki_arc_arc_easy_train.pt", ROUTER_TRAIN_SAMPLES),
            ("mom/new/data/arc_easy_validation.pt", "hidden_logprobs_wiki_arc_arc_easy_test.pt", None),
            ("mom/new/data/arc_challenge_train.pt", "hidden_logprobs_wiki_arc_arc_challenge_train.pt", ROUTER_TRAIN_SAMPLES),
            ("mom/new/data/arc_challenge_validation.pt", "hidden_logprobs_wiki_arc_arc_challenge_test.pt", None),
        ],
    },
    {
        "expert": os.path.join(EXPERT_DIR, "wiki_hellaswag.pt"),
        "pairs": [
            ("mom/new/data/hellaswag_train.pt", "hidden_logprobs_wiki_hellaswag_hellaswag_train.pt", ROUTER_TRAIN_SAMPLES),
            ("mom/new/data/hellaswag_validation.pt", "hidden_logprobs_wiki_hellaswag_hellaswag_validation.pt", None),
        ],
    },
    {
        "expert": os.path.join(EXPERT_DIR, "wiki_winogrande.pt"),
        "pairs": [
            ("mom/new/data/winogrande_train.pt", "hidden_logprobs_wiki_winogrande_winogrande_train.pt", ROUTER_TRAIN_SAMPLES),
            ("mom/new/data/winogrande_validation.pt", "hidden_logprobs_wiki_winogrande_winogrande_validation.pt", None),
        ],
    },
    {
        "expert": os.path.join(EXPERT_DIR, "wiki_boolq.pt"),
        "pairs": [
            ("mom/new/data/boolq_train.pt", "hidden_logprobs_wiki_boolq_boolq_train.pt", ROUTER_TRAIN_SAMPLES),
            ("mom/new/data/boolq_validation.pt", "hidden_logprobs_wiki_boolq_boolq_validation.pt", None),
        ],
    },
]

for spec in hidden_specs:
    for data_path, out_name, limit in spec["pairs"]:
        cmd = [
            "python",
            "-m",
            "adaptive_pruning.router.collect_hidden",
            "--expert",
            spec["expert"],
            "--model",
            LLM_MODEL,
            "--data",
            data_path,
            "--device",
            DEVICE,
            "--output",
            os.path.join(ROUTER_HIDDEN_DIR, out_name),
            "--seed",
            str(SEED),
            "--batch_size",
            str(BATCH_SIZE),
            "--hidden_layers",
            str(SPLIT_LAYER),
        ]
        if limit:
            cmd += ["--limit", str(limit)]
        subprocess.run(cmd, check=True)

### 2.3 Train router

In [ ]:
subprocess.run(
    [
        "python",
        "-m",
        "adaptive_pruning.router.train_router",
        "--input-folder",
        ROUTER_HIDDEN_DIR,
        "--output-model",
        ROUTER_MODEL_PATH,
        "--seed",
        str(SEED),
        "--layer",
        str(SPLIT_LAYER),
    ],
    check=True,
 )

## 3) Create blended expert masks

In [ ]:
# Since the benchmarks are linearly separable and the router reaches 100% test accuracy, the blended masks can be used directly with lm-eval, simplifying the workflow.

blend_specs = [
    (os.path.join(EXPERT_DIR, "arc_easy.pt"), os.path.join(EXPERT_DIR, "wiki_arc.pt")),
    (os.path.join(EXPERT_DIR, "hellaswag.pt"), os.path.join(EXPERT_DIR, "wiki_hellaswag.pt")),
    (os.path.join(EXPERT_DIR, "boolq.pt"), os.path.join(EXPERT_DIR, "wiki_boolq.pt")),
    (os.path.join(EXPERT_DIR, "winogrande.pt"), os.path.join(EXPERT_DIR, "wiki_winogrande.pt")),
]

for expert2, out_path in blend_specs:
    subprocess.run(
        [
            "python",
            "-m",
            "adaptive_pruning.router.blend_experts",
            "--expert2",
            expert2,
            "--split_layer",
            str(SPLIT_LAYER),
            "--out",
            out_path,
        ],
        check=True,
    )

## 4) Evaluate

In [ ]:
eval_all_tasks = ["boolq", "arc_easy", "arc_challenge", "winogrande", "hellaswag"]
eval_specs = [
    {
        "expert": os.path.join(EXPERT_DIR, "wiki.pt"),
        "tasks": eval_all_tasks,
    },
    {
        "expert": os.path.join(EXPERT_DIR, "wiki_arc.pt"),
        "tasks": ["arc_easy", "arc_challenge"],
    },
    {
        "expert": os.path.join(EXPERT_DIR, "wiki_hellaswag.pt"),
        "tasks": ["hellaswag"],
    },
    {
        "expert": os.path.join(EXPERT_DIR, "wiki_boolq.pt"),
        "tasks": ["boolq"],
    },
    {
        "expert": os.path.join(EXPERT_DIR, "wiki_winogrande.pt"),
        "tasks": ["winogrande"],
    },
]

for spec in eval_specs:
    subprocess.run(
        [
            "python",
            "-m",
            "adaptive_pruning.eval",
            "--llm",
            LLM_MODEL,
            "--mode",
            "flap",
            "--expert",
            spec["expert"],
            "--tasks",
            *spec["tasks"],
            "--device",
            DEVICE,
            "--batch_size",
            str(BATCH_SIZE),
            "--seed",
            str(SEED),
            "--log_file",
            EVAL_LOG_FILE,
        ],
        check=True,
    )



## Eval Dense

In [ ]:
subprocess.run(
    [
        "python",
        "-m",
        "adaptive_pruning.eval",
        "--llm",
        LLM_MODEL,
        "--mode",
        "dense",
        "--tasks",
        *eval_all_tasks,
        "--device",
        DEVICE,
        "--batch_size",
        str(BATCH_SIZE),
        "--seed",
        str(SEED),
        "--log_file",
        EVAL_LOG_FILE,
    ],
    check=True,
 )